In [ ]:
pip install torchcodec

In [ ]:
pip install librosa soundfile audioread

In [ ]:
pip install torch torchvision torchaudio

In [ ]:
pip install transformers datasets evaluate jiwer accelerate soundfile torch

In [4]:
# ---- Save as PNG ----
output_dir = Path("SPCS/graph_results")
output_dir.mkdir(parents=True, exist_ok=True)  # creates the folder if missing

table_data = [["Folder", "WAV", "TXT"]] + \
             [[f, w, t] for f, w, t in rows] + \
             [["Total", total_wav, total_txt]]

fig, ax = plt.subplots(figsize=(6, 0.4 * len(table_data) + 1))
ax.axis("off")

table = ax.table(cellText=table_data, cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)

for col in range(3):
    table[0, col].set_text_props(weight="bold")
    table[len(table_data) - 1, col].set_text_props(weight="bold")

png_path = output_dir / "file_counts.png"
plt.savefig(png_path, bbox_inches="tight", dpi=200)
plt.close(fig)
print(f"Saved table image to {png_path.resolve()}")

Saved table image to /home/lb202261599/SPCS/graph_results/file_counts.png


In [3]:
import pandas as pd
from itertools import combinations

# ---- Update these paths to match your files ----
files = {
    'train': 'SPCS/manifests/train.csv',
    'test': 'SPCS/manifests/test.csv',
    'valid': 'SPCS/manifests/valid.csv',
}

# ---- Load data ----
dfs = {name: pd.read_csv(path) for name, path in files.items()}

# Optional: restrict comparison to specific columns (e.g. an ID or filename column)
# If None, all columns are used to define a "duplicate"
check_cols = None  # e.g. ['image_path'] or ['id']

# ==========================================================
# 1. Check duplicates WITHIN each file
# ==========================================================
print("=" * 60)
print("WITHIN-FILE DUPLICATES")
print("=" * 60)

for name, df in dfs.items():
    cols = check_cols if check_cols else df.columns.tolist()
    dupe_mask = df.duplicated(subset=cols, keep=False)
    n_dupes = dupe_mask.sum()
    print(f"\n[{name}] rows: {len(df)}, duplicate rows: {n_dupes}")
    if n_dupes > 0:
        print(df[dupe_mask].sort_values(by=cols).head(10))

# ==========================================================
# 2. Check duplicates ACROSS files (pairwise overlap)
# ==========================================================
print("\n" + "=" * 60)
print("ACROSS-FILE OVERLAP (PAIRWISE)")
print("=" * 60)

for (name1, df1), (name2, df2) in combinations(dfs.items(), 2):
    cols = check_cols if check_cols else list(set(df1.columns) & set(df2.columns))

    # Merge on the comparison columns to find exact matches between the two files
    overlap = pd.merge(df1[cols], df2[cols], how='inner')
    n_overlap = len(overlap.drop_duplicates())

    print(f"\n[{name1}] vs [{name2}] -> overlapping rows: {n_overlap}")
    if n_overlap > 0:
        print(overlap.drop_duplicates().head(10))

# ==========================================================
# 3. Full combined check (any duplicate anywhere, with source tags)
# ==========================================================
print("\n" + "=" * 60)
print("ALL DUPLICATES ACROSS ALL FILES (with source)")
print("=" * 60)

tagged = []
for name, df in dfs.items():
    d = df.copy()
    d['__source'] = name
    tagged.append(d)

combined = pd.concat(tagged, ignore_index=True)
cols = check_cols if check_cols else [c for c in combined.columns if c != '__source']

all_dupes = combined[combined.duplicated(subset=cols, keep=False)]
print(f"\nTotal duplicate rows across all 3 files combined: {len(all_dupes)}")
if len(all_dupes) > 0:
    print(all_dupes.sort_values(by=cols).head(20))
    print("\nBreakdown by source:")
    print(all_dupes.groupby('__source').size())

WITHIN-FILE DUPLICATES

[train] rows: 9908, duplicate rows: 0

[test] rows: 1239, duplicate rows: 0

[valid] rows: 1239, duplicate rows: 0

ACROSS-FILE OVERLAP (PAIRWISE)

[train] vs [test] -> overlapping rows: 0

[train] vs [valid] -> overlapping rows: 0

[test] vs [valid] -> overlapping rows: 0

ALL DUPLICATES ACROSS ALL FILES (with source)

Total duplicate rows across all 3 files combined: 0


In [2]:
import pandas as pd

files = {
    'train': 'SPCS/manifests/train.csv',
    'valid': 'SPCS/manifests/valid.csv',
    'test': 'SPCS/manifests/test.csv',
}

dfs = {
    name: pd.read_csv(path)
    for name, path in files.items()
}

for name, df in dfs.items():
    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)
    
    print("Columns:", df.columns.tolist())
    print("\nFirst 20 audio paths:")
    print(df['audio'].head(20).to_string(index=False))


TRAIN
Columns: ['audio', 'text']

First 20 audio paths:
  SPCS/data/audio/010/010_Male_22_356.wav
SPCS/data/audio/012/012_Female_20_479.wav
   SPCS/data/audio/010/010_Male_22_78.wav
  SPCS/data/audio/017/017_Male_22_419.wav
  SPCS/data/audio/013/013_Male_27_284.wav
SPCS/data/audio/012/012_Female_20_563.wav
 SPCS/data/audio/006/006_Male_23_1045.wav
   SPCS/data/audio/010/010_Male_22_10.wav
SPCS/data/audio/009/009_Female_26_215.wav
  SPCS/data/audio/010/010_Male_22_498.wav
  SPCS/data/audio/020/020_Male_20_316.wav
  SPCS/data/audio/010/010_Male_22_328.wav
  SPCS/data/audio/013/013_Male_27_114.wav
  SPCS/data/audio/006/006_Male_23_429.wav
 SPCS/data/audio/003/003_Female_27_63.wav
   SPCS/data/audio/007/007_Male_23_54.wav
SPCS/data/audio/011/011_Female_21_111.wav
 SPCS/data/audio/006/006_Male_23_1048.wav
  SPCS/data/audio/010/010_Male_22_421.wav
  SPCS/data/audio/005/005_Male_26_495.wav

VALID
Columns: ['audio', 'text']

First 20 audio paths:
SPCS/data/audio/012/012_Female_20_617.wav
SPCS

In [2]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# ==========================================
# Define Project Paths
# ==========================================
audio_root = Path("SPCS/data/audio")
text_root = Path("SPCS/data/transcriptions")

# ==========================================
# Create Dataset
# ==========================================
rows = []

for wav_path in audio_root.rglob("*.wav"):
    # Get the relative path of the audio file
    rel_path = wav_path.relative_to(audio_root)

    # Find the corresponding transcription
    txt_path = text_root / rel_path.with_suffix(".txt")

    if txt_path.exists():
        text = txt_path.read_text(encoding="utf-8").strip()

        rows.append({
            "audio": str(wav_path),
            "text": text
        })

# Create DataFrame
df = pd.DataFrame(rows)

print(f"Total paired files found: {len(df)}")

# ==========================================
# Split Dataset (80% Train, 10% Validation, 10% Test)
# ==========================================
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

# ==========================================
# Save CSV Manifests
# ==========================================
manifest_dir = Path("SPCS/manifests")
manifest_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(manifest_dir / "train.csv", index=False)
valid_df.to_csv(manifest_dir / "valid.csv", index=False)
test_df.to_csv(manifest_dir / "test.csv", index=False)

# ==========================================
# Display Results
# ==========================================
print("\nCreated manifests successfully!")
print(f"Train samples      : {len(train_df)}")
print(f"Validation samples : {len(valid_df)}")
print(f"Test samples       : {len(test_df)}")
print(f"\nManifest files saved in: {manifest_dir.resolve()}")

# Preview the training data
print("\nFirst five training samples:")
display(train_df.head())

Total paired files found: 12386

Created manifests successfully!
Train samples      : 9908
Validation samples : 1239
Test samples       : 1239

Manifest files saved in: /home/lb202261599/SPCS/manifests

First five training samples:


,audio,text
6101,SPCS/data/audio/010/010_Male_22_356.wav,ge a ka dira appeal
1691,SPCS/data/audio/012/012_Female_20_479.wav,direga eng mo operation repo
6105,SPCS/data/audio/010/010_Male_22_78.wav,ikaparetše dilo tše di simple
123,SPCS/data/audio/017/017_Male_22_419.wav,bohlaba bja toropo ya Pretoria
7047,SPCS/data/audio/013/013_Male_27_284.wav,go na le category enngwe


In [ ]:
import matplotlib.pyplot as plt

splits = ["Train", "Validation", "Test"]
counts = [9908, 1239, 1239]

plt.figure(figsize=(6,4))
plt.bar(splits, counts)

plt.title("SPCS Dataset Split")
plt.xlabel("Dataset")
plt.ylabel("Number of Samples")

plt.show()

In [ ]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile huggingface_hub

# =========================================================
# IMPORTS
# =========================================================
import torch
import pandas as pd
import evaluate

from huggingface_hub import login

from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# =========================================================
# HF LOGIN (PUT YOUR NEW TOKEN HERE)
# =========================================================
# login("YOUR_HUGGINGFACE_TOKEN")

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"

# =========================================================
# LOAD DATA
# =========================================================
train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET CONVERSION
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# =========================================================
model_name = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

model.to(device)

# =========================================================
# PREPROCESSING
# =========================================================
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

train_ds = train_ds.map(
    prepare_dataset,
    remove_columns=train_ds.column_names,
    num_proc=1
)

valid_ds = valid_ds.map(
    prepare_dataset,
    remove_columns=valid_ds.column_names,
    num_proc=1
)

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# CER METRIC
# =========================================================
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {"cer": cer}

# =========================================================
# TRAINING CONFIG (BASELINE)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models/baseline_whisper",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    num_train_epochs=10,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=2,

    predict_with_generate=True,
    generation_max_length=225,

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics
)

# =========================================================
# TRAIN
# =========================================================
trainer.train()

# =========================================================
# SAVE MODEL
# =========================================================
save_path = "SPCS/models/baseline_whisper"

trainer.save_model(save_path)
processor.save_pretrained(save_path)

print("\nBaseline training complete.")
print("Model saved to:", save_path)

Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Cer
1,1.127463,0.117348,0.075882
2,0.311447,0.055063,0.024174
3,0.077726,0.038898,0.033795
4,0.006250,0.033025,0.017107


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import os
os.environ["HF_DATASETS_AUDIO_BACKEND"] = "soundfile"

In [ ]:
pip install soundfile librosa audioread

In [1]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile huggingface_hub

# =========================================================
# IMPORTS
# =========================================================
import torch
import pandas as pd
import evaluate

from huggingface_hub import login

from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# =========================================================
# HF LOGIN (PUT YOUR NEW TOKEN HERE)
# =========================================================
#login("YOUR_HUGGINGFACE_TOKEN"12345)

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"

# =========================================================
# LOAD DATA
# =========================================================
train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET CONVERSION
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# =========================================================
model_name = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

model.to(device)

# =========================================================
# PREPROCESSING
# =========================================================
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

train_ds = train_ds.map(
    prepare_dataset,
    remove_columns=train_ds.column_names,
    num_proc=1
)

valid_ds = valid_ds.map(
    prepare_dataset,
    remove_columns=valid_ds.column_names,
    num_proc=1
)

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# WER + CER METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }
# =========================================================
# TRAINING CONFIG (BASELINE)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models/baseline_whisper",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    num_train_epochs=10,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=2,

    predict_with_generate=True,
    generation_max_length=225,

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics
)

# =========================================================
# TRAIN
# =========================================================
trainer.train()

# =========================================================
# SAVE MODEL
# =========================================================
save_path = "SPCS/models/baseline_whisper"

trainer.save_model(save_path)
processor.save_pretrained(save_path)

print("\nBaseline training complete.")
print("Model saved to:", save_path)

Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Wer,Cer
1,1.125949,0.117667,0.113051,0.075252
2,0.312768,0.055298,0.058824,0.026064
3,0.080157,0.038814,0.057904,0.033130
4,0.007358,0.033485,0.032721,0.015008
5,0.003428,0.032551,0.029228,0.013924
6,0.011340,0.031368,0.027574,0.013959
7,0.000342,0.030469,0.029228,0.013959
8,0.011490,0.031127,0.028676,0.013574
9,0.000096,0.030769,0.026103,0.013084
10,0.000094,0.030826,0.027574,0.013749


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Baseline training complete.
Model saved to: SPCS/models/baseline_whisper


In [ ]:
# =========================================================
# EXPERIMENT: WHISPER SMALL + NOISE AUGMENTATION
# =========================================================

# =========================================================
# IMPORTS
# =========================================================
import torch
import pandas as pd
import numpy as np
import evaluate
import librosa
import os

from huggingface_hub import login
from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)


# =========================================================
# HF LOGIN
# =========================================================
# NOTE: replace with your own token, and set it via an env var
# (e.g. os.environ["HF_TOKEN"]) rather than hardcoding it in the script.
#login("YOUR_HUGGINGFACE_TOKEN"12345)

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"

train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# =========================================================
model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

model.to(device)

# =========================================================
# NOISE AUGMENTATION
# =========================================================
def add_noise_perturbation(audio_array, snr_db_range=(5, 20)):
    """
    Adds Gaussian white noise to the audio at a random
    signal-to-noise ratio (in dB) drawn from snr_db_range.
    Lower SNR = noisier audio.
    """
    audio_array = audio_array.astype(np.float32)

    # Signal power
    signal_power = np.mean(audio_array ** 2)
    if signal_power == 0:
        return audio_array  # silent clip, nothing to add noise to

    # Randomly choose target SNR
    snr_db = np.random.uniform(*snr_db_range)
    snr_linear = 10 ** (snr_db / 10)

    # Required noise power to hit target SNR
    noise_power = signal_power / snr_linear

    noise = np.random.normal(0, np.sqrt(noise_power), size=audio_array.shape)

    augmented_audio = audio_array + noise

    return np.clip(augmented_audio, -1.0, 1.0).astype(np.float32)

# =========================================================
# NOISE CONFIG
# =========================================================
# FIX: Use single process to avoid multiprocessing issues
NUM_PROC = 1  # Single process

# =========================================================
# TRAIN PREPROCESSING (WITH NOISE AUGMENTATION)
# =========================================================
def prepare_train_dataset(batch):
    audios = batch["audio"]

    input_features = []
    labels = []

    for i in range(len(audios)):
        audio = audios[i]

        # Apply additive noise perturbation (5-20 dB SNR)
        augmented_audio = add_noise_perturbation(
            audio["array"],
            snr_db_range=(5, 20)
        )

        feats = processor.feature_extractor(
            augmented_audio,
            sampling_rate=audio["sampling_rate"]
        ).input_features[0]

        input_features.append(feats)
        labels.append(processor.tokenizer(batch["text"][i]).input_ids)

    return {
        "input_features": input_features,
        "labels": labels
    }

# =========================================================
# VALIDATION PREPROCESSING (CLEAN AUDIO)
# =========================================================
def prepare_valid_dataset(batch):
    audios = batch["audio"]

    input_features = []
    labels = []

    for i in range(len(audios)):
        audio = audios[i]

        feats = processor.feature_extractor(
            audio["array"],
            sampling_rate=audio["sampling_rate"]
        ).input_features[0]

        input_features.append(feats)
        labels.append(processor.tokenizer(batch["text"][i]).input_ids)

    return {
        "input_features": input_features,
        "labels": labels
    }

# =========================================================
# FAST DATA LOADING (FIXED - USING SINGLE PROCESS)
# =========================================================
print("Preprocessing training data...")
train_ds = train_ds.map(
    prepare_train_dataset,
    batched=True,
    batch_size=8,
    num_proc=1,  # Single process
    remove_columns=train_ds.column_names
)

print("Preprocessing validation data...")
valid_ds = valid_ds.map(
    prepare_valid_dataset,
    batched=True,
    batch_size=8,
    num_proc=1,  # Single process
    remove_columns=valid_ds.column_names
)

print("Data preprocessing complete!")

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

# =========================================================
# TRAINING CONFIGURATION (FIXED - removed group_by_length)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models/whisper_small_noise_aug",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    num_train_epochs=10,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=2,

    predict_with_generate=True,
    generation_max_length=225,

    dataloader_num_workers=0,


    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics
)

# =========================================================
# TRAIN
# =========================================================
print("Starting training...")
trainer.train()

# =========================================================
# SAVE MODEL
# =========================================================
save_path = "SPCS/models/whisper_small_noise_aug"

trainer.save_model(save_path)
processor.save_pretrained(save_path)

print("\nExperiment complete: Whisper + Noise Augmentation (5-20 dB SNR)")
print("Saved to:", save_path)

# =========================================================
# OPTIONAL: TEST THE MODEL
# =========================================================
print("\nTesting model on validation set...")
results = trainer.evaluate()
print(f"Final WER: {results['eval_wer']:.4f}")
print(f"Final CER: {results['eval_cer']:.4f}")

Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Preprocessing training data...


Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

In [ ]:
# =========================================================
# EXPERIMENT: WHISPER SMALL + SPEED PERTURBATION
# =========================================================

# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa huggingface_hub

# =========================================================
# IMPORTS
# =========================================================
import torch
import pandas as pd
import numpy as np
import evaluate
import librosa
import os

from huggingface_hub import login
from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)


# =========================================================
# HF LOGIN
# =========================================================
#login("YOUR_HUGGINGFACE_TOKEN"12345)

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"

train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# =========================================================
model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

model.to(device)

# =========================================================
# SPEED PERTURBATION AUGMENTATION
# =========================================================
def add_speed_perturbation(audio_array, speed_range=(0.9, 1.1)):
    """
    Randomly changes speech speed without affecting pitch.
    """
    speed_factor = np.random.uniform(*speed_range)

    # Time-stretch audio (speed change)
    augmented_audio = librosa.effects.time_stretch(
        audio_array.astype(np.float32),
        rate=speed_factor
    )

    return np.clip(augmented_audio, -1.0, 1.0).astype(np.float32)

# =========================================================
# SPEED CONFIG
# =========================================================
# FIX: Use single process to avoid multiprocessing issues
NUM_PROC = 1  # Single process

# =========================================================
# TRAIN PREPROCESSING (WITH SPEED AUGMENTATION)
# =========================================================
def prepare_train_dataset(batch):
    audios = batch["audio"]

    input_features = []
    labels = []

    for i in range(len(audios)):
        audio = audios[i]

        # Apply speed perturbation (0.9×–1.1×)
        augmented_audio = add_speed_perturbation(
            audio["array"],
            speed_range=(0.9, 1.1)
        )

        feats = processor.feature_extractor(
            augmented_audio,
            sampling_rate=audio["sampling_rate"]
        ).input_features[0]

        input_features.append(feats)
        labels.append(processor.tokenizer(batch["text"][i]).input_ids)

    return {
        "input_features": input_features,
        "labels": labels
    }

# =========================================================
# VALIDATION PREPROCESSING (CLEAN AUDIO)
# =========================================================
def prepare_valid_dataset(batch):
    audios = batch["audio"]

    input_features = []
    labels = []

    for i in range(len(audios)):
        audio = audios[i]

        feats = processor.feature_extractor(
            audio["array"],
            sampling_rate=audio["sampling_rate"]
        ).input_features[0]

        input_features.append(feats)
        labels.append(processor.tokenizer(batch["text"][i]).input_ids)

    return {
        "input_features": input_features,
        "labels": labels
    }

# =========================================================
# FAST DATA LOADING (FIXED - USING SINGLE PROCESS)
# =========================================================
print("Preprocessing training data...")
train_ds = train_ds.map(
    prepare_train_dataset,
    batched=True,
    batch_size=8,
    num_proc=1,  # Single process
    remove_columns=train_ds.column_names
)

print("Preprocessing validation data...")
valid_ds = valid_ds.map(
    prepare_valid_dataset,
    batched=True,
    batch_size=8,
    num_proc=1,  # Single process
    remove_columns=valid_ds.column_names
)

print("Data preprocessing complete!")

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

# =========================================================
# TRAINING CONFIGURATION (FIXED - removed group_by_length)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models/whisper_small_speed_aug",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    num_train_epochs=10,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=2,

    predict_with_generate=True,
    generation_max_length=225,

    dataloader_num_workers=0,
    

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics
)

# =========================================================
# TRAIN
# =========================================================
print("Starting training...")
trainer.train()

# =========================================================
# SAVE MODEL
# =========================================================
save_path = "SPCS/models/whisper_small_speed_aug"

trainer.save_model(save_path)
processor.save_pretrained(save_path)

print("\nExperiment complete: Whisper + Speed Perturbation (0.9–1.1×)")
print("Saved to:", save_path)

# =========================================================
# OPTIONAL: TEST THE MODEL
# =========================================================
print("\nTesting model on validation set...")
results = trainer.evaluate()
print(f"Final WER: {results['eval_wer']:.4f}")
print(f"Final CER: {results['eval_cer']:.4f}")

In [1]:
# =========================================================
# EXPERIMENT: WHISPER SMALL + PITCH AUGMENTATION
# =========================================================

# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa huggingface_hub
 
# =========================================================
# IMPORTS
# =========================================================
import torch
import pandas as pd
import numpy as np
import evaluate
import librosa
import os

from huggingface_hub import login
from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# Disable tokenizers parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# =========================================================
# HF LOGIN
# =========================================================
# NOTE: Don't hardcode your HF token in a script you might share or commit.
# Prefer: login(os.environ["HF_TOKEN"])  (set HF_TOKEN as an env var / secret)
login("YOUR_HUGGINGFACE_TOKEN")

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"

train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# =========================================================
model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

model.to(device)

# =========================================================
# PITCH SHIFT AUGMENTATION
# =========================================================
def add_pitch_shift(audio_array, sr, n_steps_range=(-2, 2)):
    """
    Randomly shifts pitch up or down (in semitones) without changing speed/duration.
    n_steps_range is in semitones, e.g. (-2, 2) means shift by up to +/- 2 semitones.
    """
    n_steps = np.random.uniform(*n_steps_range)

    augmented_audio = librosa.effects.pitch_shift(
        audio_array.astype(np.float32),
        sr=sr,
        n_steps=n_steps
    )

    return np.clip(augmented_audio, -1.0, 1.0).astype(np.float32)

# =========================================================
# PITCH CONFIG
# =========================================================
# FIX: Use single process to avoid multiprocessing issues
NUM_PROC = 1  # Single process

# =========================================================
# TRAIN PREPROCESSING (WITH PITCH AUGMENTATION)
# =========================================================
def prepare_train_dataset(batch):
    audios = batch["audio"]

    input_features = []
    labels = []

    for i in range(len(audios)):
        audio = audios[i]

        # Apply pitch shift (+/- 2 semitones)
        augmented_audio = add_pitch_shift(
            audio["array"],
            sr=audio["sampling_rate"],
            n_steps_range=(-2, 2)
        )

        feats = processor.feature_extractor(
            augmented_audio,
            sampling_rate=audio["sampling_rate"]
        ).input_features[0]

        input_features.append(feats)
        labels.append(processor.tokenizer(batch["text"][i]).input_ids)

    return {
        "input_features": input_features,
        "labels": labels
    }

# =========================================================
# VALIDATION PREPROCESSING (CLEAN AUDIO)
# =========================================================
def prepare_valid_dataset(batch):
    audios = batch["audio"]

    input_features = []
    labels = []

    for i in range(len(audios)):
        audio = audios[i]

        feats = processor.feature_extractor(
            audio["array"],
            sampling_rate=audio["sampling_rate"]
        ).input_features[0]

        input_features.append(feats)
        labels.append(processor.tokenizer(batch["text"][i]).input_ids)

    return {
        "input_features": input_features,
        "labels": labels
    }

# =========================================================
# FAST DATA LOADING (FIXED - USING SINGLE PROCESS)
# =========================================================
print("Preprocessing training data...")
train_ds = train_ds.map(
    prepare_train_dataset,
    batched=True,
    batch_size=8,
    num_proc=1,  # Single process
    remove_columns=train_ds.column_names
)

print("Preprocessing validation data...")
valid_ds = valid_ds.map(
    prepare_valid_dataset,
    batched=True,
    batch_size=8,
    num_proc=1,  # Single process
    remove_columns=valid_ds.column_names
)

print("Data preprocessing complete!")

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

# =========================================================
# TRAINING CONFIGURATION
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models/whisper_small_pitch_aug",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    num_train_epochs=10,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=2,

    predict_with_generate=True,
    generation_max_length=225,

    dataloader_num_workers=0,

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics
)

# =========================================================
# TRAIN
# =========================================================
print("Starting training...")
trainer.train()

# =========================================================
# SAVE MODEL
# =========================================================
save_path = "SPCS/models/whisper_small_pitch_aug"

trainer.save_model(save_path)
processor.save_pretrained(save_path)

print("\nExperiment complete: Whisper + Pitch Augmentation (+/- 2 semitones)")
print("Saved to:", save_path)

# =========================================================
# OPTIONAL: TEST THE MODEL
# =========================================================
print("\nTesting model on validation set...")
results = trainer.evaluate()
print(f"Final WER: {results['eval_wer']:.4f}")
print(f"Final CER: {results['eval_cer']:.4f}")

Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Preprocessing training data...


Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Preprocessing validation data...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Experiment complete: Whisper + Pitch Augmentation (+/- 2 semitones)
Saved to: SPCS/models/whisper_small_pitch_aug

Testing model on validation set...


Training Loss,Validation Loss,Epoch,Wer,Cer
0.000016,0.013772,10,0.006250,0.003359


Final WER: 0.0063
Final CER: 0.0034


In [5]:
# =========================================================
# EXPERIMENT: WHISPER SMALL + SPEC AUGMENT
# =========================================================

# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile huggingface_hub

# =========================================================
# IMPORTS
# =========================================================
import torch
import pandas as pd
import numpy as np
import evaluate
import os

from huggingface_hub import login
from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# Disable tokenizers parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# =========================================================
# LOGIN
# =========================================================
#login("YOUR_HUGGINGFACE_TOKEN")

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# DATA
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"

train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL
# =========================================================
model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

model.to(device)

# =========================================================
# SPEC AUGMENT FUNCTIONS
# =========================================================
def time_mask(spec, T=30, num_masks=2):
    spec = spec.copy()
    num_frames = spec.shape[0]

    for _ in range(num_masks):
        t = np.random.randint(0, T)
        t0 = np.random.randint(0, max(1, num_frames - t))
        spec[t0:t0 + t, :] = 0

    return spec


def freq_mask(spec, F=15, num_masks=2):
    spec = spec.copy()
    num_mel = spec.shape[1]

    for _ in range(num_masks):
        f = np.random.randint(0, F)
        f0 = np.random.randint(0, max(1, num_mel - f))
        spec[:, f0:f0 + f] = 0

    return spec

# =========================================================
# APPLY SPEC AUGMENT (ON FEATURES)
# =========================================================
def apply_spec_augment(features):
    features = time_mask(features, T=30, num_masks=2)
    features = freq_mask(features, F=15, num_masks=2)
    return features

# =========================================================
# PREPROCESSING (FIXED - USING SINGLE PROCESS)
# =========================================================
NUM_PROC = 1  # Single process to avoid multiprocessing issues

def prepare_train_dataset(batch):
    audios = batch["audio"]

    input_features = []
    labels = []

    for i in range(len(audios)):
        audio = audios[i]

        feats = processor.feature_extractor(
            audio["array"],
            sampling_rate=audio["sampling_rate"]
        ).input_features[0]

        # APPLY SPEC AUGMENT HERE
        feats = apply_spec_augment(np.array(feats))

        input_features.append(feats)
        labels.append(processor.tokenizer(batch["text"][i]).input_ids)

    return {
        "input_features": input_features,
        "labels": labels
    }

def prepare_valid_dataset(batch):
    audios = batch["audio"]

    input_features = []
    labels = []

    for i in range(len(audios)):
        audio = audios[i]

        feats = processor.feature_extractor(
            audio["array"],
            sampling_rate=audio["sampling_rate"]
        ).input_features[0]

        input_features.append(feats)
        labels.append(processor.tokenizer(batch["text"][i]).input_ids)

    return {
        "input_features": input_features,
        "labels": labels
    }

# =========================================================
# MAP DATASET (FIXED - USING SINGLE PROCESS)
# =========================================================
print("Preprocessing training data...")
train_ds = train_ds.map(
    prepare_train_dataset,
    batched=True,
    batch_size=8,
    num_proc=1,  # Single process
    remove_columns=train_ds.column_names
)

print("Preprocessing validation data...")
valid_ds = valid_ds.map(
    prepare_valid_dataset,
    batched=True,
    batch_size=8,
    num_proc=1,  # Single process
    remove_columns=valid_ds.column_names
)

print("Data preprocessing complete!")

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

# =========================================================
# TRAINING (FIXED - removed group_by_length)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models/whisper_small_specaug",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    num_train_epochs=10,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",  
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=2,

    predict_with_generate=True,
    generation_max_length=225,

    dataloader_num_workers=0,  # FIX: Set to 0 to avoid multiprocessing issues
    # REMOVED: group_by_length=True  # This parameter doesn't exist in Seq2SeqTrainingArguments

    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics
)

# =========================================================
# TRAIN
# =========================================================
print("Starting training...")
trainer.train()

# =========================================================
# SAVE
# =========================================================
save_path = "SPCS/models/whisper_small_specaug"

trainer.save_model(save_path)
processor.save_pretrained(save_path)

print("\nExperiment complete: Whisper + SpecAugment")
print("Saved to:", save_path)

# =========================================================
# OPTIONAL: TEST THE MODEL
# =========================================================
print("\nTesting model on validation set...")
results = trainer.evaluate()
print(f"Final WER: {results['eval_wer']:.4f}")
print(f"Final CER: {results['eval_cer']:.4f}")

Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Preprocessing training data...


Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Preprocessing validation data...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Experiment complete: Whisper + SpecAugment
Saved to: SPCS/models/whisper_small_specaug

Testing model on validation set...


Training Loss,Validation Loss,Epoch,Wer,Cer
0.000015,0.019800,10,0.010662,0.005143


Final WER: 0.0107
Final CER: 0.0051


In [6]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

audio_root = Path("SPCS/data/audio")
text_root = Path("SPCS/data/transcriptions")

# 1. Build df (your original pairing code)
rows = []
for wav_path in audio_root.rglob("*.wav"):
    rel_path = wav_path.relative_to(audio_root)
    txt_path = text_root / rel_path.with_suffix(".txt")
    if txt_path.exists():
        rows.append({"audio": str(wav_path),
                     "text": txt_path.read_text(encoding="utf-8").strip()})

df = pd.DataFrame(rows)
print(f"Total paired files: {len(df)}")

# 2. Add the speaker column BEFORE splitting
df["speaker"] = df["audio"].apply(
    lambda p: Path(p).relative_to(audio_root).parts[0]
)
print(f"Total speakers: {df['speaker'].nunique()}")

# 3. Now the speaker-aware split will work
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(gss.split(df, groups=df["speaker"]))
train_df, temp_df = df.iloc[train_idx], df.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
valid_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["speaker"]))
valid_df, test_df = temp_df.iloc[valid_idx], temp_df.iloc[test_idx]

for name, d in [("Train", train_df), ("Valid", valid_df), ("Test", test_df)]:
    print(f"{name}: {len(d)} files, {d['speaker'].nunique()} speakers")

Total paired files: 12386
Total speakers: 20
Train: 9703 files, 16 speakers
Valid: 1230 files, 2 speakers
Test: 1453 files, 2 speakers
